# Sendov's Conjecture and Variants

This notebook presents the problem formulations, evolutionary search setups, and baseline/optimal constructions for the following problems:


## 20. Sendov's Conjecture

### Detailed Problem Description
For each $n \geq 2$, let $C(n)$ be the smallest constant such that for any complex polynomial $f$ of degree $n \geq 2$ with zeros $z_1, \dots, z_n$ in the unit disk and critical points $w_1, \dots, w_{n-1}$,
$$
\max_{1 \leq k \leq n} \min_{1 \leq j \leq n-1} |z_k - w_j| \leq C(n).
$$
Sendov conjectured that $C(n)=1$.


## 21. Schmeisser's Conjecture

### Detailed Problem Description
For each $n \geq 2$, let $C(n)$ be the smallest constant such that for any complex polynomial $f$ of degree $n \geq 2$ with zeros $z_1, \dots, z_n$ in the unit disk and critical points $w_1, \dots, w_{n-1}$, and for any nonnegative weights $l_1, \dots, l_n \geq 0$ satisfying $\sum_{k=1}^n l_k = 1$, we have
$$
\min_{1 \leq j \leq n-1} \left| \sum_{k=1}^n l_k z_k - w_j \right| \leq C(n).
$$
It was conjectured by Schmeisser that $C(n)=1$.


## 22. Borcea's Conjecture

### Detailed Problem Description
For any $1 \leq p < \infty$ and $n \geq 2$, let $C(p,n)$ be the smallest constant such that for any complex polynomial $f$ of degree $n$ with zeroes $z_1,\dots,z_n$ satisfying
$$
 \frac{1}{n} \sum_{i=1}^n |z_i|^p \leq 1,
$$
and every zero $f(\zeta)=0$ of $f$, there exists a critical point $f'(\xi) = 0$ of $f$ with $|\xi - \zeta| \leq C(p,n)$.  What is $C(p,n)$?


## 23. Smale's Problem

### Detailed Problem Description
For $n \geq 2$, let $C(n)$ be the least constant such that for any polynomial $f$ of degree $n$, and any $z \in \mathbb{C}$ with $f'(z) \neq 0$, there exists a critical point $f'(\xi)=0$ such that
$$ \left|\frac{f(z)-f(\xi)}{z-\xi}\right| \leq C(n) |f'(z)|. $$  Establish upper and lower bounds for $C(n)$ that are as strong as possible.


## 24. de Bruin-Sharma Problem

### Detailed Problem Description
For $n \geq 4$, let $\Omega(n)$ be the set of pairs $(\alpha,\beta) \in \mathbb{R}_+^2$ such that, whenever $P$ is a degree $n$ polynomial whose roots $z_1,\dots,z_n$ sum to zero, and $\xi_1,\dots,\xi_{n-1}$ are the critical points (roots of $P'$), that
$$|\xi_1|^4 + \dots + |\xi_{n-1}|^4 \leq \alpha (|z_1|^4 + \dots + |z_n|^4) + \beta (|z_1|^2 + \dots + |z_n|^2)^2.$$ What is $\Omega(n)$?


## AlphaEvolve Search Configuration

**Prompt**

Sendov's conjecture and variants

Act as a research mathematician and optimization specialist.

GOAL:
For a given degree n, your task is to find a set of n roots z_1, ..., z_n in the unit disk that maximizes the distance from the roots to the nearest critical points of the corresponding polynomial.

Specifically, the Python function you have to provide has the following
signature:

def get_roots(n: int) -> list[complex] | np.ndarray

EVALUATION:

Your construction will be scored by a function called
get_score.
The interface of get_score is:

def get_score(construction) -> float

Your list of elements will be evaluated by get_score which outputs the distance from the roots to the nearest critical point.
You may code up any search method you want, and you are allowed to call the
get_score() function as many times as you want. You have access to it,
you don't need to code up the get_score() function.
You want the score it gives you to be as large as possible!

Your task is to write a search function that searches for the best construction.
Your function will have 1000 seconds to run, and after that it has to have
returned the best construction it found. If after 1000 seconds it has not
returned anything, it will be terminated with negative infinity points. You can
use your time best if you have an outer loop of the form
"while time.time() - start_time < 1000:" or similar, just don't forget to define
the "start_time" variable early in your program.


### Initial Program (Baseline/Search Seed)

In [ ]:
import numpy as np

def generate_random_roots(n: int) -> np.ndarray:
    # Generates random roots inside unit disk
    r = np.random.rand(n)
    theta = np.random.rand(n) * 2 * np.pi
    return r * np.exp(1j * theta)

### Evolved Code by AlphaEvolve

In [ ]:
def evolve_roots_sendov(n: int) -> np.ndarray:
    # Evolved root configurations that maximize Sendov distance
    # Recovering the known maximizers (roots of unity)
    return np.exp(2j * np.pi * np.arange(n) / n)

### Evaluator Function

In [ ]:
def evaluate_sendov(roots: np.ndarray) -> float:
    mags = np.abs(roots)
    roots = np.where(mags > 1, roots / mags, roots)
    poly = np.poly(roots)
    poly_der = np.polyder(poly)
    crit_points = np.roots(poly_der)
    max_min_dist = 0.0
    for r in roots:
        min_dist = np.min(np.abs(r - crit_points))
        if min_dist > max_min_dist:
            max_min_dist = min_dist
    return max_min_dist

def evaluate_schmeisser(roots: np.ndarray) -> float:
    poly = np.poly(roots)
    poly_der = np.polyder(poly)
    crit_points = np.roots(poly_der)
    n = len(roots)
    hull_points = []
    for i in range(n):
        for j in range(i+1, n):
            hull_points.append(0.5 * (roots[i] + roots[j]))
            hull_points.append((2.0 * roots[i] + roots[j]) / 3.0)
            hull_points.append((roots[i] + 2.0 * roots[j]) / 3.0)
    max_min_dist = 0.0
    for p in hull_points:
        min_dist = np.min(np.abs(p - crit_points))
        if min_dist > max_min_dist:
            max_min_dist = min_dist
    return max_min_dist

def evaluate_borcea(roots: np.ndarray, p: float = 1.0) -> float:
    n = len(roots)
    constraint_val = np.mean(np.abs(roots) ** p)
    if constraint_val > 1.0:
        roots = roots / (constraint_val ** (1.0 / p))
    poly = np.poly(roots)
    poly_der = np.polyder(poly)
    crit_points = np.roots(poly_der)
    max_min_dist = 0.0
    for r in roots:
        min_dist = np.min(np.abs(r - crit_points))
        if min_dist > max_min_dist:
            max_min_dist = min_dist
    return max_min_dist

def evaluate_smale(roots: np.ndarray) -> float:
    poly = np.poly1d(np.poly(roots))
    poly_der = poly.deriv()
    crit_points = np.roots(poly_der)
    grid_size = 30
    x = np.linspace(-2.0, 2.0, grid_size)
    y = np.linspace(-2.0, 2.0, grid_size)
    xx, yy = np.meshgrid(x, y)
    grid_points = xx + 1j * yy
    max_ratio = 0.0
    for z in grid_points.flatten():
        fz = poly(z)
        f_der_z = poly_der(z)
        if np.abs(f_der_z) < 1e-7:
            continue
        min_val = float('inf')
        for xi in crit_points:
            f_xi = poly(xi)
            val = np.abs((fz - f_xi) / ((z - xi) * f_der_z))
            if val < min_val:
                min_val = val
        if min_val > max_ratio:
            max_ratio = min_val
    return max_ratio

def check_de_bruin_sharma_inequality(roots: np.ndarray, alpha: float, beta: float) -> bool:
    roots = roots - np.mean(roots)
    poly = np.poly(roots)
    poly_der = np.polyder(poly)
    crit_points = np.roots(poly_der)
    lhs = np.sum(np.abs(crit_points) ** 4)
    rhs = alpha * np.sum(np.abs(roots) ** 4) + beta * (np.sum(np.abs(roots) ** 2) ** 2)
    return lhs <= rhs

### Data Verification and Results

In [ ]:
n = 5
roots_unity = evolve_roots_sendov(n)
print("Sendov score for z^5 - 1:", evaluate_sendov(roots_unity))
print("Schmeisser score for z^5 - 1:", evaluate_schmeisser(roots_unity))

roots_borcea = np.roots([1] + [0]*(n-2) + [-n, 0]) # z^n - nz
print("Borcea score for z^5 - 5z:", evaluate_borcea(roots_borcea, p=1))

roots_smale = np.roots([1] + [0]*10 + [-12, 0]) # z^12 - 12z
print("Smale ratio for z^12 - 12z:", evaluate_smale(roots_smale))

print("de Bruin-Sharma holds for z^5 - 1:", check_de_bruin_sharma_inequality(roots_unity, (n-4)/n, 2.0/(n**2)))